# Notebook 03 — Procedimientos almacenados y cursores

Tercer sub-bloque del Tema 06. Un **procedimiento almacenado** empaqueta una secuencia de acciones (modificar tablas, validar, registrar) en un objeto reutilizable que se invoca con **`CALL`**. A diferencia de una función, **no** se usa dentro de un `SELECT` — está pensado para **efectos secundarios**, no para devolver un valor a una query.

También vemos **cursores explícitos**, una construcción que conviene reconocer (aparece en código legacy) aunque en código moderno casi siempre tiene un reemplazo más limpio con `FOR ... IN SELECT`.

**Contenido de este notebook:**

- [Setup](#setup)
- [Procedimiento vs función — la diferencia clave](#procedimiento-vs-función--la-diferencia-clave)
- [`CREATE PROCEDURE` y `CALL`](#create-procedure-y-call)
- [Parámetros `IN`, `OUT`, `INOUT`](#parámetros-in-out-inout)
- [Cursores explícitos — declaración y uso](#cursores-explícitos--declaración-y-uso)
- [Cursores vs `FOR` loops sobre queries](#cursores-vs-for-loops-sobre-queries)
- [¿Cuándo usar procedimiento o SQL puro?](#cuándo-usar-procedimiento-o-sql-puro)

## Setup

In [ ]:
# Setup — instala JupySQL si hace falta (Colab trae ipython-sql, no JupySQL).
import importlib.util, subprocess, sys
if importlib.util.find_spec("jupysql") is None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "ipython-sql"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupysql"], check=True)
    print("⚠ JupySQL instalado. REINICIA el kernel (Entorno de ejecución → Reiniciar sesión)")
    print("  y vuelve a correr esta celda y las siguientes.")
else:
    print("✓ JupySQL listo.")

In [ ]:
%load_ext sql

from sqlalchemy import create_engine

# Reemplaza con tus valores del Tema 01
AURORA_HOST     = "aurora-mod4.cluster-xxxxx.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "TU_PASSWORD_AQUI"
AURORA_DATABASE = "northwind"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}"
)

%sql engine
# Devuelve cada query como DataFrame de pandas (mejor render en Colab, y
# el resultado es directamente manipulable con pandas).
%config SqlMagic.autopandas = True

## Procedimiento vs función — la diferencia clave

Una **función** devuelve un valor y se usa **dentro de un `SELECT`** (`SELECT mi_funcion(args)`) — como las funciones agregadas, de texto y de fecha que viste en el Tema 05. Un **procedimiento** es distinto: se invoca con **`CALL`**, no entrega un valor para usar en una query, y existe para ejecutar **efectos secundarios**.

| Aspecto | Función | Procedimiento (`CREATE PROCEDURE`) |
|---|---|---|
| Cómo se invoca | `SELECT mi_funcion(args)` | `CALL mi_procedimiento(args);` |
| ¿Usable en `SELECT`/`WHERE`/`JOIN`? | ✅ Sí | ❌ No |
| Caso de uso típico | Calcular y devolver un valor | Ejecutar una serie de acciones con efectos secundarios |

**La regla:** si tu lógica es "calcula algo y devuélvemelo" → función. Si es "haz una serie de acciones (modificar tablas, validar, registrar)" → **procedimiento**.

Casos típicos de procedimiento: refrescar agregados pre-calculados, importar un batch de datos con validaciones, ejecutar mantenimiento periódico.

## `CREATE PROCEDURE` y `CALL`

Sintaxis:

```sql
CREATE PROCEDURE nombre(parametros) AS $$
BEGIN
    -- código
END;
$$ LANGUAGE plpgsql;

-- Invocar:
CALL nombre(args);
```

Nota la ausencia de `RETURNS tipo` — un procedimiento puede devolver valores a través de parámetros `OUT`, pero no "devuelve" en el sentido funcional.

### La cláusula `LANGUAGE` — ¿siempre `plpgsql`?

La línea `$$ LANGUAGE plpgsql;` no es decorativa: le dice a PostgreSQL **en qué lenguaje está escrito el cuerpo** entre los `$$ ... $$`. Es **obligatoria** — no hay valor por defecto. Si la omites:

```sql
CREATE FUNCTION f() RETURNS int AS $$ SELECT 1 $$;
-- ERROR: no language specified
```

Pero **no tiene que ser `plpgsql`**. Hay dos lenguajes que usarás en la práctica:

| `LANGUAGE` | Cuándo usarlo | Qué soporta |
|---|---|---|
| **`sql`** | El cuerpo es **solo SQL** (uno o varios `SELECT`/`INSERT`/`UPDATE`) | Sentencias SQL planas. Más ligero y el planner puede *inlinearlo* |
| **`plpgsql`** | Necesitas **lógica procedural** | Variables, `IF`/`CASE`, `LOOP`/`FOR`, `RAISE`, `EXCEPTION`, `BEGIN…END` con flujo |

**La regla:** si tu procedimiento solo encadena sentencias SQL → `LANGUAGE sql`. En cuanto necesites variables, condicionales, loops o manejo de errores → `LANGUAGE plpgsql` (el único de los dos que entiende esas construcciones). En este tema casi todo es `plpgsql` precisamente porque el tema *trata* de esa lógica procedural — un `IF` o un `RAISE` dentro de un cuerpo `LANGUAGE sql` fallaría.

#### Los lenguajes instalados (y los que se pueden instalar)

Por defecto, una base PostgreSQL trae cuatro lenguajes (`SELECT lanname FROM pg_language;`):

| Lenguaje | Para qué |
|---|---|
| `sql` | Funciones/procedimientos de SQL plano (lo de arriba) |
| `plpgsql` | El lenguaje procedural de PostgreSQL (este tema) |
| `c` | Funciones compiladas en C — para extensiones de alto rendimiento |
| `internal` | Funciones implementadas dentro del propio motor (no las escribes tú) |

**¿Se pueden instalar más?** Sí. PostgreSQL permite **lenguajes procedurales adicionales** vía `CREATE EXTENSION`, para escribir el cuerpo de una función en otro lenguaje (Python, Perl, JavaScript…). En un PostgreSQL self-managed puedes añadir `plpython3u`, `plperl`, `pltcl`, `plv8` (JavaScript), etc.

> :information_source: **En Aurora/RDS el set está restringido por AWS.** En tu cluster, `SELECT name FROM pg_available_extensions WHERE name LIKE 'pl%';` devuelve los que SÍ puedes instalar: **`plperl`**, **`pltcl`**, **`plv8`** (+ `plcoffee`/`plls`, variantes de JS) y el profiler `plprofiler`. Nota que **`plpython` no aparece**: AWS no permite lenguajes "untrusted" (acceso al filesystem del host) en bases administradas. Para usar uno, primero `CREATE EXTENSION plv8;` y luego `... $$ LANGUAGE plv8;`. Para BI, con `sql` + `plpgsql` cubres prácticamente todo.


In [ ]:
%%sql
DROP TABLE IF EXISTS log_eventos;
CREATE TEMP TABLE log_eventos (tipo TEXT, mensaje TEXT, momento TIMESTAMP DEFAULT now());

CREATE OR REPLACE PROCEDURE registrar_evento(
    p_tipo    TEXT,
    p_mensaje TEXT
) AS $$
BEGIN
    -- efecto secundario: el procedimiento registra el evento en una tabla
    INSERT INTO log_eventos (tipo, mensaje) VALUES (p_tipo, p_mensaje);
END;
$$ LANGUAGE plpgsql;

In [ ]:
%%sql
CALL registrar_evento('INFO', 'Procedimiento ejecutado correctamente');
CALL registrar_evento('WARN', 'Algo merece atención');

SELECT * FROM log_eventos;

## Parámetros `IN`, `OUT`, `INOUT`

Los parámetros de un procedimiento (o función) pueden tener tres modos:

- **`IN`** (default) — solo entrada. Lo más común. Pasas un valor al procedimiento.
- **`OUT`** — solo salida. El procedimiento le asigna un valor que el llamador recibe.
- **`INOUT`** — entrada y salida. Pasas un valor, el procedimiento lo modifica.

**Cómo recibir valores de un `OUT`:** desde SQL puro, los argumentos `OUT` se ven en el resultado del `CALL`.

```sql
CREATE PROCEDURE dividir(
    p_a   NUMERIC,
    p_b   NUMERIC,
    OUT p_resultado NUMERIC,
    OUT p_resto     NUMERIC
) AS $$
BEGIN
    p_resultado := p_a / p_b;
    p_resto     := p_a % p_b;
END;
$$ LANGUAGE plpgsql;

CALL dividir(17, 5, NULL, NULL);     -- los NULL son placeholders para los OUT
-- Resultado: p_resultado=3.4..., p_resto=2
```

In [ ]:
%%sql
CREATE OR REPLACE PROCEDURE dividir(
    p_a NUMERIC,
    p_b NUMERIC,
    OUT p_cociente NUMERIC,
    OUT p_resto    NUMERIC
) AS $$
BEGIN
    p_cociente := TRUNC(p_a / p_b);
    p_resto    := p_a - p_cociente * p_b;
END;
$$ LANGUAGE plpgsql;

In [ ]:
%%sql
CALL dividir(17, 5, NULL, NULL);

## Cursores explícitos — declaración y uso

Un **cursor** es un puntero a una query que se procesa fila a fila. PL/pgSQL tiene cursores **implícitos** (los del `FOR ... IN SELECT` del Notebook 02) y **explícitos**:

```sql
DECLARE
    mi_cursor CURSOR FOR SELECT col FROM tabla WHERE ...;
BEGIN
    OPEN mi_cursor;
    LOOP
        FETCH mi_cursor INTO variable;
        EXIT WHEN NOT FOUND;
        -- usar variable
    END LOOP;
    CLOSE mi_cursor;
END;
```

Las operaciones son:

- **`OPEN`** — ejecuta la query y posiciona el cursor antes de la primera fila.
- **`FETCH ... INTO`** — avanza una fila y guarda los valores en variables.
- **`CLOSE`** — libera recursos.

**`FOUND`** se setea automáticamente: `false` cuando `FETCH` no trae más filas (final del set).

In [ ]:
%%sql
DROP TABLE IF EXISTS salida;
CREATE TEMP TABLE salida (n INTEGER, categoria TEXT);

DO $$
DECLARE
    cur_categorias CURSOR FOR
        SELECT category_name FROM northwind_dwh.dim_product
        GROUP BY category_name ORDER BY 1;
    cat TEXT;
    n   INTEGER := 0;
BEGIN
    OPEN cur_categorias;
    LOOP
        FETCH cur_categorias INTO cat;
        EXIT WHEN NOT FOUND;
        n := n + 1;
        INSERT INTO salida VALUES (n, cat);
    END LOOP;
    CLOSE cur_categorias;
END
$$;

SELECT * FROM salida;

## Cursores vs `FOR` loops sobre queries

El mismo trabajo del ejemplo anterior se puede expresar mucho más limpio con un `FOR` loop sobre query:

```sql
DO $$
DECLARE
    n INTEGER := 0;
BEGIN
    FOR cat IN SELECT category_name FROM ... LOOP
        n := n + 1;
        INSERT INTO salida VALUES (n, cat.category_name);
    END LOOP;
END
$$;
```

Por debajo, el `FOR ... IN SELECT` crea y maneja un cursor implícito. La diferencia es que tú no tienes que escribir `OPEN`/`FETCH`/`CLOSE`.

**Cuándo conviene cursor explícito:**

- Cuando necesitas pasarlo como argumento a otra función (los cursores se pueden devolver de funciones — patrón llamado *refcursor*).
- Para procesamiento con `FETCH` parametrizado (`FETCH 10 FROM cur` lee 10 a la vez).
- Para movimientos no-secuenciales: `FETCH PRIOR`, `FETCH FIRST`, `FETCH LAST` (con cursores `SCROLL`).
- Compatibilidad con código heredado.

**Cuándo NO usar cursor (la mayoría):** prefiere `FOR ... IN SELECT`. Es más legible y maneja la limpieza automáticamente.

**Lo más importante:** si tu intención era hacer una transformación masiva (`UPDATE`, `INSERT INTO ... SELECT`), **ningún cursor es la respuesta correcta**. Una sola query SQL es 100× más rápida y mucho más simple.

## ¿Cuándo usar procedimiento o SQL puro?

Tabla de decisión rápida:

| Caso | Herramienta correcta |
|---|---|
| Transformación masiva (`UPDATE ... FROM`, `INSERT ... SELECT`) | **SQL puro — sin procedimiento** |
| Serie de acciones con efectos secundarios (refrescar agregados, cargar batch con validaciones) | **Procedimiento** |
| Iterar fila por fila para procesar cada una distinta | **`FOR ... IN SELECT`** dentro de un procedimiento |
| Procesar fila por fila con `OPEN`/`FETCH` explícitos | **Cursor — solo si tienes una razón específica** |

**Regla mnemónica:**

1. Si SQL puro lo expresa → SQL puro (rapidísimo, optimizable).
2. Si necesitas efectos secundarios o una secuencia de pasos → procedimiento.
3. Cursor solo cuando hay razón muy específica (procesamiento en lotes con `FETCH N`, scrollable, compatibilidad).

In [ ]:
%%sql
-- Limpieza de objetos creados en este notebook
DROP PROCEDURE IF EXISTS registrar_evento(TEXT, TEXT);
DROP PROCEDURE IF EXISTS dividir(NUMERIC, NUMERIC);
DROP TABLE     IF EXISTS log_eventos;

## Cierre

Lo que cubriste:

| Tema | Construcción clave |
|---|---|
| Crear procedimiento | `CREATE PROCEDURE nombre(params) AS $$ ... $$ LANGUAGE plpgsql;` |
| Invocar | `CALL nombre(args);` — no `SELECT` |
| Eliminar | `DROP PROCEDURE nombre(tipos);` |
| Parámetros con dirección | `IN` (default), `OUT`, `INOUT` |
| Cursor explícito | `DECLARE c CURSOR FOR ...; OPEN c; FETCH c INTO v; CLOSE c;` |
| Cursor implícito | `FOR row IN SELECT ... LOOP ... END LOOP;` — la opción limpia |

El siguiente notebook (**04 — Práctica**) consolida lo de los notebooks anteriores con ejercicios graduales sobre Northwind.

---

<p align="center">
<a href="02_control_de_flujo.ipynb">← Anterior: Notebook 02</a> | <a href="Readme.md">Volver al índice</a> | <a href="04_practica.ipynb">Siguiente: Notebook 04 — Práctica →</a>
</p>